<a href="https://colab.research.google.com/github/KinzaaSheikh/llm_engineering_notes/blob/main/Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install -qU transformers[sentencepiece]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 39.4 MB/s eta 0:00:00


## Basics of the Architecture

In [14]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

classifier("The weather today is extremely hot!")

# to me it shows unnecessary error and also the output is wrong, positivity score of 99% to a rather negative or neutral statement

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9973548650741577}]

In [10]:
classifier(["I have been waiting for this moment my whole life!", "I hate this so much!"])

[{'label': 'POSITIVE', 'score': 0.9984257221221924},
 {'label': 'NEGATIVE', 'score': 0.9994558691978455}]

### Zero-shot Classification


In [11]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification")
classifier(
    "This book is about dragons",
    candidate_labels=["books", "fantasy", "sci-fi"],
)

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

{'sequence': 'This book is about dragons',
 'labels': ['fantasy', 'books', 'sci-fi'],
 'scores': [0.9800259470939636, 0.01812211237847805, 0.0018519547302275896]}

Called zero-shot because there is no need to fine tuning, but still results can be different from what you want to use cases must be limited.


### Text Generation

In [12]:
from transformers import pipeline

generator = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-360M")
generator("This book was about", max_length=30, num_return_sequences=2,)

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  724MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=30) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': 'This book was about the second century AD, but it is about the first few hundred years of the 20th century.\n\nWe know that the United States was a tiny country (1/2 the size of the present-day United States) when the French Revolution was in full swing, and it was a republic. It was a republic with a president, but there were no military or police forces. There were a few lawyers, but they were hired by the government, not by their clients. The economy was mostly agricultural, with a few towns and cities, but no major manufacturing. There were no railroads, no interstate highways, no telephones, no television, no radio, no computers or even electric light. The people worked hard for a living, but they were not slaves. In short, it was a country where people lived the way the Roman Empire lived.\n\nThe United States was not a democracy, but it was a republic, and the Republic was not democratic. The United States was a republic, but it was not a democracy.\n\nThe U

## Behind the Scenes of the Model

In [15]:
# download pretrained model

from transformers import AutoTokenizer, AutoModel

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModel.from_pretrained(checkpoint)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
pre_classifier.weight | UNEXPECTED |  | 
pre_classifier.bias   | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 
classifier.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
raw_inputs = [
    "I've been reading books my whole life.",
    "I love it so much1",
]
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")
print(inputs)

{'input_ids': tensor([[ 101, 1045, 1005, 2310, 2042, 3752, 2808, 2026, 2878, 2166, 1012,  102],
        [ 101, 1045, 2293, 2009, 2061, 2172, 2487,  102,    0,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]])}


In [17]:
outputs = model(**inputs)
print(outputs.last_hidden_state.shape)

torch.Size([2, 12, 768])


## Creating a Transformer

In [19]:
from transformers import AutoModel, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

model = AutoModel.from_pretrained("bert-base-cased")


# automodel can instantiate the most appropriate model itself in case undefined

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
# function for encoding text into integers

encoded_input = tokenizer("Hello, I'm a single sentence!")
print(encoded_input)

{'input_ids': [101, 8667, 117, 146, 112, 182, 170, 1423, 5650, 106, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [21]:
# decoding it back to text

tokenizer.decode(encoded_input["input_ids"])

"[CLS] Hello, I ' m a single sentence! [SEP]"

#### Tokenization Exercises

In [22]:
encoded_input = tokenizer("Hello world")
print(encoded_input)

{'input_ids': [101, 8667, 1362, 102], 'token_type_ids': [0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1]}


In [23]:
tokenizer.decode(encoded_input["input_ids"])

'[CLS] Hello world [SEP]'

In [24]:
encoded_input = tokenizer("Transformers")
print(encoded_input)

{'input_ids': [101, 25267, 102], 'token_type_ids': [0, 0, 0], 'attention_mask': [1, 1, 1]}


In [25]:
tokenizer.decode(encoded_input["input_ids"])

'[CLS] Transformers [SEP]'

In [26]:
encoded_input = tokenizer("unbelievable")
print(encoded_input)

{'input_ids': [101, 8362, 26438, 102], 'token_type_ids': [0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1]}


In [27]:
tokenizer.decode(encoded_input["input_ids"])

'[CLS] unbelievable [SEP]'

In [28]:
encoded_input = tokenizer("ChatGPT is amazing.")
print(encoded_input)

{'input_ids': [101, 24705, 1204, 17095, 1942, 1110, 6929, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [29]:
tokenizer.decode(encoded_input["input_ids"])

'[CLS] ChatGPT is amazing. [SEP]'

In [30]:
encoded_input = tokenizer("The mitochondria is the powerhouse of the cell.")
print(encoded_input)

{'input_ids': [101, 1109, 26410, 9962, 16838, 3464, 1110, 1103, 1540, 3255, 1104, 1103, 2765, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [31]:
tokenizer.decode(encoded_input["input_ids"])

'[CLS] The mitochondria is the powerhouse of the cell. [SEP]'